In [5]:
# ===== CELL 1: Cleaning pipeline =====
import pandas as pd, numpy as np
from datetime import timedelta

sales = pd.read_csv("data/raw/sales_transactions.csv", parse_dates=['InvoiceDate'])
print(f"Before cleaning: {len(sales):,} rows")

# Step 1: Drop guest checkouts (need CustomerID for segmentation/churn)
sales['CustomerID'] = sales['CustomerID'].astype(str).replace('nan', '')
sales_clean = sales[sales['CustomerID'] != ''].copy()
print(f"  After dropping guests:    {len(sales_clean):,}  (-{len(sales)-len(sales_clean):,})")

# Step 2: Drop cancellations
sales_clean = sales_clean[~sales_clean['Invoice'].astype(str).str.startswith('C')]
print(f"  After dropping cancels:   {len(sales_clean):,}")

# Step 3: Drop non-positive quantity / price
sales_clean = sales_clean[(sales_clean['Quantity'] > 0) & (sales_clean['UnitPrice'] > 0)]
print(f"  After Q/P filter:         {len(sales_clean):,}")

# Step 4: Drop missing description
sales_clean = sales_clean.dropna(subset=['Description'])

# Step 5: Drop exact duplicates
sales_clean = sales_clean.drop_duplicates()
print(f"  After dedup:              {len(sales_clean):,}")

# Step 6: Winsorize at 99th percentile
for col in ['Quantity', 'UnitPrice']:
    cap = sales_clean[col].quantile(0.99)
    sales_clean[col] = sales_clean[col].clip(upper=cap)

sales_clean['TotalPrice'] = (sales_clean['Quantity'] * sales_clean['UnitPrice']).round(2)
sales_clean.to_csv("data/processed/sales_cleaned.csv", index=False)
print(f"\n✅ Final cleaned: {len(sales_clean):,} rows → data/processed/sales_cleaned.csv")




Before cleaning: 357,645 rows
  After dropping guests:    339,763  (-17,882)
  After dropping cancels:   329,530
  After Q/P filter:         329,530
  After dedup:              329,530

✅ Final cleaned: 329,530 rows → data/processed/sales_cleaned.csv


In [7]:
# Load customers
import pandas as pd
# Load RFM dataset
rfm = pd.read_csv("data/processed/rfm_features.csv")

# Load customers dataset

customers = pd.read_csv("data/raw/customers_master.csv")

# Fix CustomerID format
rfm['CustomerID'] = rfm['CustomerID'].astype(float).astype(int).astype(str)
customers['CustomerID'] = customers['CustomerID'].astype(float).astype(int).astype(str)

# Remove old duplicate demographic columns if already present
cols_to_remove = [
    'AgeGroup_x','Gender_x','LoyaltyTier_x','PreferredChannel_x','Country_x',
    'AgeGroup_y','Gender_y','LoyaltyTier_y','PreferredChannel_y','Country_y',
    'AgeGroup','Gender','LoyaltyTier','PreferredChannel','Country'
]

rfm = rfm.drop(columns=[c for c in cols_to_remove if c in rfm.columns], errors='ignore')

# Fresh clean merge
rfm = rfm.merge(
    customers[['CustomerID','AgeGroup','Gender','LoyaltyTier','PreferredChannel','Country']],
    on='CustomerID',
    how='left'
)

# Save
rfm.to_csv("data/processed/rfm_features.csv", index=False)

# Check output
print(rfm.head())
print(f"\n✅ Final RFM dataset: {rfm.shape}")

  CustomerID  Recency  Frequency  Monetary         AOV  UniqueProducts  \
0      12001      131          8   4453.46  212.069524              19   
1      12002       80         10   2831.72  128.714545              19   
2      12003      406          1    107.74   53.870000               2   
3      12004       63         12   4107.15  128.348438              19   
4      12005      254         14   7089.92  141.798400              30   

   UniqueCategories  Tenure     AvgQty  R_Score  F_Score  M_Score  RFM_Score  \
0                 8     477  12.142857        3        3        4        334   
1                 8     529   2.272727        3        3        3        333   
2                 2       0   2.500000        1        1        1        111   
3                 7     469   2.718750        4        4        3        443   
4                 8     214   3.780000        2        4        4        244   

  AgeGroup Gender LoyaltyTier PreferredChannel         Country  
0    45-5

In [8]:
# ===== CELL 3: Rolling time-series features (per SKU) =====
ts = (sales_clean.groupby([pd.Grouper(key='InvoiceDate', freq='D'), 'StockCode'])
      .agg(units_sold=('Quantity', 'sum'),
           revenue   =('TotalPrice', 'sum'),
           n_invoices=('Invoice', 'nunique'))
      .reset_index())

ts = ts.sort_values(['StockCode','InvoiceDate'])

# Rolling means
for w in [7, 14, 30]:
    ts[f'units_roll_{w}']   = ts.groupby('StockCode')['units_sold'].transform(
                                lambda x: x.rolling(w, min_periods=1).mean()).round(2)
    ts[f'revenue_roll_{w}'] = ts.groupby('StockCode')['revenue'].transform(
                                lambda x: x.rolling(w, min_periods=1).mean()).round(2)

# Lag features
for lag in [1, 7, 14]:
    ts[f'units_lag_{lag}'] = ts.groupby('StockCode')['units_sold'].shift(lag)

# Calendar features
ts['day_of_week'] = ts['InvoiceDate'].dt.dayofweek
ts['day_of_month']= ts['InvoiceDate'].dt.day
ts['month']       = ts['InvoiceDate'].dt.month
ts['quarter']     = ts['InvoiceDate'].dt.quarter
ts['is_weekend']  = (ts['day_of_week'] >= 5).astype(int)
ts['is_dec']      = (ts['month'] == 12).astype(int)

ts.to_csv("data/processed/timeseries_daily.csv", index=False)
print(f"✅ Time-series features: {len(ts):,} rows × {len(ts.columns)} cols")

✅ Time-series features: 66,446 rows × 20 cols


In [10]:
!pip install great_expectations

   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/5.0 MB ? eta -:--:--
   ------------ --------------------------- 1.6/5.0 MB 6.2 MB/s eta 0:00:01
   ----------------------- ---------------- 2.9/5.0 MB 5.8 MB/s eta 0:00:01
   ------------------------------- -------- 3.9/5.0 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 5.0/5.0 MB 5.4 MB/s  0:00:01

   ------------- -------------------------- 1/3 [marshmallow]
   -------------------------- ------------- 2/3 [great_expectations]
   -------------------------- ------------- 2/3 [great_expectations]
   -------------------------- ------------- 2/3 [great_expectations]
   -------------------------- ------------- 2/3 [great_expectations]
   -------------------------- ------------- 2/3 [great_expectations]
   -------------------------- ------------- 2/3 [great_expectations]
   -------------------------- ------------- 2/3 [great_expectations]
   ------------

In [13]:
import pandas as pd

def validate(df, name):

    checks = []

    if 'CustomerID' in df.columns:
        checks.append(df['CustomerID'].notnull().all())

    if 'Quantity' in df.columns:
        checks.append(df['Quantity'].between(1, 10000).all())

    if 'UnitPrice' in df.columns:
        checks.append(df['UnitPrice'].between(0.01, 1000).all())

    if 'TotalPrice' in df.columns:
        checks.append(df['TotalPrice'].notnull().all())

    if 'Invoice' in df.columns:
        checks.append(df['Invoice'].astype(str).str.match(r'^\d+$').all())

    passed = sum(checks)

    print(f"{name:25s}: {passed}/{len(checks)} expectations passed")

    return all(checks)

print("VALIDATION RESULTS")
print("="*50)

validate(sales_clean, "sales_cleaned")
validate(rfm, "rfm_features")

VALIDATION RESULTS
sales_cleaned            : 5/5 expectations passed
rfm_features             : 1/1 expectations passed


True